# Task 1.1: Core Contribution / Architecture of PVM

**Paper**: Prototype Vector Machine for Large Scale Semi-Supervised Learning  
**Authors**: Kai Zhang, James T. Kwok, Bahram Parvin  
**Venue**: ICML 2009  
**Student**: Ritesh Patil (230056)

## Step-by-Step Method Description

### Step 1: Problem Setup — Graph-Based Semi-Supervised Learning Framework

- **Description**: The method starts with a dataset containing `l` labeled samples and `u` unlabeled samples (total `n = l + u`). The goal is to predict labels for all data points by solving an optimization problem that balances three objectives: (i) smoothness of predicted labels over a graph connecting similar points, (ii) consistency with known labels, and (iii) regularization on unlabeled predictions.
- **Reference**: Equation (1), Section 2. The optimization framework is: `min tr(f^T S f) + C1 * L(f_l, Y_l) + C2 * ||f_u||^2_F`.
- **Purpose**: This establishes the general SSL framework that PVM will make scalable. The matrix `S` (graph Laplacian) encodes the manifold structure, and `f` is the matrix of predicted labels for all points.

### Step 2: Low-Rank Approximation Prototypes — Approximating the Graph Regularizer

- **Description**: Instead of computing the full `n x n` kernel matrix `K`, PVM selects `m` prototype points `{u_i}` and uses the Nystrom method to approximate `K ≈ E * W^(-1) * E^T`, where `W` is the `m x m` kernel matrix on prototypes and `E` is the `n x m` cross-similarity matrix between all data and prototypes. This makes the graph regularization term `f^T S f` computable without ever forming the full kernel matrix.
- **Reference**: Equation (4), Section 3.1. The approximation `K ≈ K_nm * K_mm^(-1) * K_nm^T` is the Nystrom formula.
- **Purpose**: This solves the first scalability bottleneck — the `n x n` kernel matrix computation. By using only `m` prototypes (where `m << n`), the cost drops from O(n^2) storage and O(n^3) computation to O(n*m).

### Step 3: Label-Reconstruction Prototypes — Compressing the Model

- **Description**: The representer theorem (Belkin et al., 2006) says the optimal model expands over ALL `n` data points as `f(x) = sum_i alpha_i * K(x, x_i)`. PVM replaces this with a compact model using `k` prototypes: `g(x) = sum_i f_i * K(x, v_i)`, where `{v_i}` are label-reconstruction prototypes. In matrix form, `f = H * f_v`, where `H` is an `n x k` cross-similarity matrix and `f_v` is the `k x c` label matrix for prototypes.
- **Reference**: Equations (5) and (6), Section 3.2. The model approximation and the reconstruction formula are defined here.
- **Purpose**: This solves the second scalability bottleneck — the model size. Instead of optimizing over `n` variables (one label per data point), PVM optimizes over only `k` prototype labels, drastically reducing the problem size.

### Step 4: Prototype Selection via K-Means — Unified Selection Criterion

- **Description**: The paper shows that for the Gaussian kernel, selecting prototypes to minimize the information loss (measured via KL-divergence between basis functions) reduces exactly to the k-means clustering objective. This means both types of prototypes can be chosen as k-means cluster centers computed on the full dataset. This is a key theoretical insight — one simple operation (k-means) optimally serves both roles.
- **Reference**: Equation (8), Section 3.2. The KL-divergence between Gaussian kernels simplifies to `(1/4h^2) * ||x_i - v_j||^2`, which is the squared distance minimized by k-means.
- **Purpose**: This provides a principled, unified, and computationally cheap method to select prototypes. Rather than needing separate procedures for each prototype type, a single k-means run suffices.

### Step 5: Reformulated Optimization — PVM with L2 Loss (PVM(1))

- **Description**: By substituting the prototype-based approximations (Steps 2 and 3) into the original SSL objective (Step 1) with an L2 loss function, PVM obtains a new, smaller optimization problem. The solution has a closed form: `f_v* = (H^T S H + C1 * H_l^T H_l + C2 * H_u^T H_u)^(-1) * H_l^T * Y_l`. This involves inverting a `k x k` matrix instead of an `n x n` one.
- **Reference**: Equation (9), Section 4.1.
- **Purpose**: This produces the final PVM model for multi-class problems. The closed-form solution means no iterative optimization is needed, and the overall complexity is O((m+k)^2 * n), which is nearly linear in `n`.

### Step 6: Reformulated Optimization — PVM with Hinge Loss (PVM(2))

- **Description**: For binary classification, PVM can also use the SVM hinge loss. By plugging the prototype approximations into the SVM formulation and deriving the Lagrangian dual, PVM obtains a QP (quadratic programming) problem with only `l` dual variables (number of labeled points), similar to a standard SVM. The matrix `Q = H_l * A^(-1) * H_l^T ⊙ Y_l * Y_l^T` replaces the standard kernel matrix in the SVM dual.
- **Reference**: Equations (10)-(11), Section 4.2.
- **Purpose**: This enables PVM to leverage efficient SVM solvers and inherit SVM's sparse solutions. After solving, prototype labels are recovered via the KKT condition.

### Step 7: Prediction for All Data Points

- **Description**: Once the optimal prototype labels `f_v*` are obtained (from either Step 5 or Step 6), the predicted labels for all unlabeled points are computed as `f_u = H_u * f_v*`. For new test points not seen during training, their labels can also be computed by evaluating kernel similarities with the prototypes. This makes PVM inductive (unlike many graph-based methods which are transductive).
- **Reference**: Section 4.2, final paragraph.
- **Purpose**: This produces the final output — predicted class labels for all unlabeled (and potentially new) data points.

## Final Summary Sentence

This paper solves the problem of **scalability in graph-based semi-supervised learning**, which traditionally scales cubically with data size due to the need to compute and manipulate large kernel matrices and models that span all data points. The authors claim their approach is better than existing alternatives because it uses **prototype vectors selected via k-means clustering** to simultaneously approximate both the graph regularizer (via Nystrom low-rank approximation) and the prediction model (via label reconstruction), achieving **near-linear scaling** O((m+k)^2 * n) while maintaining classification accuracy competitive with full graph-based methods.